# Random Forest và XGBoost — baseline dữ liệu thật

Dự đoán `label_withdrawn` tại đầu ngày 28. Dùng bảng `feature_28d.csv` và `split.csv` chung với LR. Hai mô hình dùng cùng 33 cột gốc và nhánh `tree` (49 feature) của `preprocessor.joblib` do nhóm đã fit trên train. Chỉ đánh giá dev ở ngưỡng cố định 0,5; test dành cho nghiệm thu cuối.

Notebook gọi mã dùng lại trong `src/models/baseline_rf_xgb_real.py`. Nhánh PCA, chọn tham số/ngưỡng và benchmark là các bước tiếp theo. Kết quả này cùng cohort/split/nhãn và đặc tả feature gốc với LR. RF/XGBoost dùng nhánh `tree` (số gốc), LR dùng nhánh `linear` (chuẩn hóa); không xem chênh lệch metric là do duy nhất thuật toán.


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.models.baseline_rf_xgb_real import run
DATASET = ROOT / 'dataset'
RESULTS = ROOT / 'results'
assert DATASET.is_dir(), f'Không tìm thấy thư mục {DATASET}'


In [2]:
metrics, importance, raw_features = run(DATASET, RESULTS)
print(f'{len(raw_features)} feature gốc; hai mô hình có cùng số feature sau mã hóa: {metrics.n_features.iloc[0]}')
print('Feature gốc:', raw_features)
metrics


/home/tyler-vu/miniconda3/envs/big_data/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OrdinalEncoder from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/tyler-vu/miniconda3/envs/big_data/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/tyler-vu/miniconda3/envs/big_data/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator FunctionTr

33 feature gốc; hai mô hình có cùng số feature sau mã hóa: 49
Feature gốc: ['code_module', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'disability', 'num_of_prev_attempts', 'studied_credits', 'date_registration', 'imd_missing', 'date_registration_missing', 'total_clicks', 'active_days', 'clicks_per_active_day', 'clicks_w1', 'clicks_w2', 'clicks_w3', 'clicks_w4', 'click_trend', 'n_distinct_sites', 'n_distinct_activity_types', 'first_active_day', 'last_active_day', 'days_since_last_active', 'longest_gap_days', 'no_activity_flag', 'n_assess_due', 'n_assess_submitted', 'assess_submit_rate', 'avg_score', 'weighted_score', 'no_assess_flag']


,model,split,n_train,n_dev,n_features,threshold,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,random_forest,dev,19264,4131,49,0.5,0.422139,0.299601,0.350467,0.694753,0.353224,3072,308,526,225
1,xgboost,dev,19264,4131,49,0.5,0.310954,0.585885,0.406279,0.704632,0.378043,2405,975,311,440


In [3]:
importance.groupby('model', group_keys=False).head(15)


,model,feature,importance
36,random_forest,num_raw__date_registration,0.067339
43,random_forest,num_raw__avg_score,0.055542
25,random_forest,num_raw__clicks_per_active_day,0.054068
37,random_forest,num_raw__click_trend,0.054034
23,random_forest,num_raw__total_clicks,0.053034
28,random_forest,num_raw__clicks_w3,0.047480
26,random_forest,num_raw__clicks_w1,0.046912
27,random_forest,num_raw__clicks_w2,0.046267
29,random_forest,num_raw__clicks_w4,0.044607
30,random_forest,num_raw__n_distinct_sites,0.042955


Kết quả được ghi tại `results/rf_xgb_baseline_dev.csv` và `results/rf_xgb_feature_importance.csv`. Các số trên dev chỉ là baseline 100 cây cho mỗi mô hình trên 49 feature từ preprocessor tree chung ở ngưỡng 0,5; không dùng test để chọn cấu hình. Tên feature trong bảng importance là tên sau mã hóa; các mức của một cột phân loại xuất hiện thành nhiều dòng.
